# 3.0 Build Individual Logistic Regression Models

**Objective:** Determine which single feature produces the highest prediction accuracy.

We will build 16 separate Logistic Regression models. Each model uses:
- One independent variable (predictor)
- One dependent variable (target: `outcome`)


## 3.1 Understand the Modelling Approach

Each model uses one feature to predict the insurance claim outcome.

**Examples**

**Model 1**
\[
outcome = f(age)
\]

**Model 2**
\[
outcome = f(gender)
\]

**Model 3**
\[
outcome = f(driving\_experience)
\]

We will use the `logit()` function from Statsmodels.


## 3.2 Understand the Logistic Regression Formula

```python
logit('outcome ~ age', data=car_insurance_clean)
```

| Component | Meaning |
|---|---|
| `outcome` | Dependent variable (what we predict) |
| `~` | Explained by |
| `age` | Independent variable (predictor) |
| `data` | Dataset used for modelling |

In simple terms, we are asking:

**Can a customer's age predict whether they will make an insurance claim?**

For categorical variables, use `C()` to explicitly tell Statsmodels to treat the feature as categorical:

```python
logit('outcome ~ C(vehicle_type)', data=car_insurance_clean)
```

This is useful when a feature contains text categories such as `sedan` and `sports car`.


## 3.3 Build a Logistic Regression Model for Each Feature

Instead of manually creating 16 models, we can use a `for` loop.

The process is:

1. Loop through every feature in `features`.
2. Construct a Logistic Regression formula.
3. Fit the model using the dataset.
4. Generate predicted probabilities.
5. Convert probabilities into binary predictions.
6. Calculate the accuracy score.
7. Store the results for comparison.

### Prerequisites

This notebook expects `car_insurance_clean` to be the cleaned dataset and `features` to contain the 16 predictor column names. The target column, `outcome`, should be binary (0/1).


### 3.3.1 Import the Required Libraries

If you have already imported these libraries in an earlier notebook, this cell can be skipped.


In [ ]:
import pandas as pd
from statsmodels.formula.api import logit


### 3.3.2 Define the Features

The following list follows the 16 features in this exercise. Make sure the names match the columns in `car_insurance_clean`.


In [ ]:
features = [
    'age',
    'gender',
    'driving_experience',
    'education',
    'income',
    'credit_score',
    'vehicle_ownership',
    'vehicle_year',
    'married',
    'children',
    'postal_code',
    'annual_mileage',
    'vehicle_type',
    'speeding_violations',
    'duis',
    'past_accidents'
]


### 3.3.3 Create an Empty Dictionary

We need somewhere to store the accuracy of each model.


In [ ]:
# Create an empty dictionary to store model accuracy
accuracies = {}


### 3.3.4 Train Individual Models

The loop trains one Logistic Regression model for each feature and stores its training accuracy.

> **Categorical variables:** If a feature is stored as text categories, wrap it in `C()` in the formula (for example, `outcome ~ C(vehicle_type)`). The code below follows the direct formula approach used for the reported results.


In [ ]:
# Loop through each candidate feature
for feature in features:

    # Create the Logistic Regression formula
    formula = f'outcome ~ {feature}'

    # Fit the Logistic Regression model
    model = logit(formula, data=car_insurance_clean).fit()

    # Generate predicted probabilities
    predictions = model.predict(car_insurance_clean)

    # Convert probabilities into binary predictions
    predicted_classes = (predictions >= 0.5).astype(int)

    # Calculate accuracy
    accuracy = (predicted_classes == car_insurance_clean['outcome']).mean()

    # Store the accuracy
    accuracies[feature] = accuracy


### Understanding the Important Code

**1. Constructing the formula**

```python
formula = f'outcome ~ {feature}'
```

The f-string dynamically inserts each feature into the formula.

- First iteration: `outcome ~ age`
- Second iteration: `outcome ~ gender`
- Third iteration: `outcome ~ driving_experience`

**2. Generating predictions**

```python
predictions = model.predict(car_insurance_clean)
```

Logistic Regression returns predicted probabilities between 0 and 1.

**3. Converting probabilities into classes**

```python
predicted_classes = (predictions >= 0.5).astype(int)
```

We use 0.5 as the classification threshold.

| Probability | Predicted Class |
|---:|---:|
| 0.85 | 1 |
| 0.25 | 0 |
| 0.63 | 1 |

**4. Calculating accuracy**

```python
accuracy = (predicted_classes == car_insurance_clean['outcome']).mean()
```

Accuracy measures the proportion of correct predictions.

\[
\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}
\]

For example, if the model correctly predicts 8,000 out of 10,000 customers:

\[
\text{Accuracy} = \frac{8000}{10000} = 0.80
\]

This means the model correctly predicts **80% of the customers**.

**5. Storing the accuracy**

```python
accuracies[feature] = accuracy
```

This stores each feature and its corresponding accuracy in the dictionary.


## 3.4 Compare the Model Accuracy

Once all 16 models have been trained, convert the dictionary into a DataFrame and sort the results by accuracy in descending order.


In [ ]:
# Convert the accuracy dictionary into a DataFrame
accuracy_df = pd.DataFrame(
    list(accuracies.items()),
    columns=['features', 'accuracy']
)

# Sort models by accuracy in descending order
accuracy_df = accuracy_df.sort_values(
    by='accuracy',
    ascending=False
).reset_index(drop=True)

# Display the results
accuracy_df


### Reported Results

The following table shows the results from the exercise.

| Rank | Feature | Accuracy |
|---:|---|---:|
| 1 | driving_experience | 0.7771 |
| 2 | age | 0.7747 |
| 3 | income | 0.7425 |
| 4 | vehicle_ownership | 0.7351 |
| 5 | credit_score | 0.7053 |
| 6 | annual_mileage | 0.6904 |
| 7 | gender | 0.6867 |
| 8 | education | 0.6867 |
| 9 | vehicle_year | 0.6867 |
| 10 | married | 0.6867 |
| 11 | children | 0.6867 |
| 12 | postal_code | 0.6867 |
| 13 | vehicle_type | 0.6867 |
| 14 | speeding_violations | 0.6867 |
| 15 | duis | 0.6867 |
| 16 | past_accidents | 0.6867 |

The displayed table is the reported output from the exercise; running the notebook on a different dataset or with different preprocessing may produce different values.


### Important Modelling Consideration

In the code above, we train and evaluate each model using the same 10,000 records.

This measures **training accuracy**, not how well the model generalizes to unseen customers.

For this exercise, we are following the straightforward approach to identify the highest-accuracy feature. In a real production project, we should use a **train-test split** or **cross-validation** to obtain a more reliable performance estimate.


## 3.5 Interpretation of Logistic Regression Modelling

A total of 16 individual Logistic Regression models were trained, with each model using one feature to predict the insurance claim outcome (`outcome`). Predicted probabilities were converted into binary classes using a classification threshold of 0.5, and the accuracy of each model was calculated to evaluate its predictive performance.

The results showed that `driving_experience` achieved the highest accuracy of **77.71%**, followed closely by `age` at **77.47%**. This indicates that driving experience and age provide the strongest predictive performance among the individual features.

On the other hand, several features achieved an accuracy of **68.67%**, which was equivalent to the majority-class baseline. This suggests that these features did not improve prediction accuracy when used individually.

Overall, the modelling process allows us to compare the predictive performance of each feature and identify `driving_experience` as the best-performing single feature for predicting car insurance claim outcomes.

**Key takeaway:** By training and evaluating individual Logistic Regression models, we identified `driving_experience` as the best-performing feature, with an accuracy of **77.71%**.
